# ДЗ 31. Прогнозирование загрузки метро (Metro Interstate Traffic Volume)

Цель — построить модель, предсказывающую объём трафика (`traffic_volume`) на ближайшую неделю вперёд (24*7 точек), используя данные [Metro Interstate Traffic Volume Dataset](https://archive.ics.uci.edu/ml/datasets/Metro+Interstate+Traffic+Volume).

Несмотря на название датасета (это на самом деле данные о трафике на шоссе I-94 между Миннеаполисом и Сент-Полом), в задании он используется как прокси для загрузки метро — методология прогнозирования от этого не меняется.

План работы:
1. Загрузка данных и базовый EDA
2. Чистка ряда: дубликаты, выравнивание временной сетки, интерполяция пропусков
3. Отбор релевантной истории + генерация признаков (день недели, час)
4. Baseline-прогноз (среднее по `weekday`/`hour`)
5. Модели: SARIMA, SARIMAX с температурой, градиентный бустинг на признаках
6. Финальный прогноз на следующую неделю с доверительным интервалом


## 1. Загрузка данных и базовый EDA

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams["figure.figsize"] = (14, 4)
plt.rcParams["axes.grid"] = True

pd.set_option("display.max_columns", 50)

RANDOM_STATE = 42


In [ ]:
raw = pd.read_csv("data/Metro_Interstate_Traffic_Volume.csv")
print(raw.shape)
raw.head()


In [ ]:
# По заданию нужны только 4 столбца
df = raw[["date_time", "holiday", "temp", "traffic_volume"]].copy()
df["date_time"] = pd.to_datetime(df["date_time"])
df.info()


In [ ]:
df.describe(include="all")


### Пропуски и базовые аномалии


In [ ]:
print("Пропуски по столбцам:")
print(df.isna().sum())
print()
print("Уникальные значения holiday:", df["holiday"].nunique())
print(df["holiday"].value_counts().head(15))


In [ ]:
# temp = 0 Kelvin выглядит как явный выброс/ошибка датчика (0K это abs. zero, нереально)
print("Число записей с temp == 0:", (df["temp"] == 0).sum())
df.loc[df["temp"] == 0, "temp"] = np.nan


### Дубликаты по времени

В датасете каждая строка соответствует часу, но часто на один и тот же `date_time` приходится
несколько записей (например, из-за нескольких значений `weather_description` для одного часа
в исходных данных NOAA). Так как нам нужны только 4 колонки, такие дубликаты по `date_time`
надо схлопнуть.


In [ ]:
dup_mask = df.duplicated(subset="date_time", keep=False)
print(f"Строк с повторяющимся date_time: {dup_mask.sum()} из {len(df)}")
df[dup_mask].sort_values("date_time").head(6)


In [ ]:
# Убираем полные дубликаты, а затем схлопываем оставшиеся дубликаты по date_time
# (берём первую запись - traffic_volume/temp в пределах часа отличаются незначительно)
df = df.drop_duplicates()
df = df.drop_duplicates(subset="date_time", keep="first")
df = df.sort_values("date_time").reset_index(drop=True)
print(df.shape)


### Равномерность временной сетки

Проверим, действительно ли все точки идут с шагом в 1 час.


In [ ]:
diffs = df["date_time"].diff().dropna()
print(diffs.value_counts().head(10))
print()
print("Доля шагов != 1 час:", (diffs != pd.Timedelta(hours=1)).mean())


In [ ]:
gap_sizes = diffs[diffs > pd.Timedelta(hours=1)].sort_values(ascending=False)
print("Крупнейшие пропуски во времени:")
print(gap_sizes.head(10))
print()
print(f"Всего 'дыр' длиннее 1 часа: {len(gap_sizes)}")
print(f"Суммарно пропущено часов: {(gap_sizes / pd.Timedelta(hours=1) - 1).sum():.0f}")


Есть как дубликаты, так и заметные пропуски во времени (вплоть до многодневных). Выровняем сетку
до строго часовой при помощи `pd.date_range`, переиндексируем и заполним пропуски линейной
интерполяцией.


In [ ]:
df = df.set_index("date_time")

full_index = pd.date_range(df.index.min(), df.index.max(), freq="h")
print(f"Было точек: {len(df)}, должно быть при равномерной сетке: {len(full_index)}")

df = df.reindex(full_index)
df.index.name = "date_time"


In [ ]:
# holiday - категориальный признак, интерполировать нельзя - заполняем "None" (нет праздника)
df["holiday"] = df["holiday"].fillna("None")

# temp и traffic_volume - непрерывные ряды, линейная интерполяция
df["temp"] = df["temp"].interpolate(method="linear")
df["traffic_volume"] = df["traffic_volume"].interpolate(method="linear")

print(df.isna().sum())


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(df.index, df["traffic_volume"], lw=0.5)
axes[0].set_title("traffic_volume, весь период")
axes[1].plot(df.index, df["temp"], lw=0.5, color="tab:orange")
axes[1].set_title("temp, весь период")
plt.tight_layout()
plt.show()


### Недельная и суточная сезонность

Посмотрим на характерный профиль загрузки по дням недели и часам на небольшом фрагменте ряда.


In [ ]:
sample = df.loc["2018-08-01":"2018-08-21"]
plt.figure(figsize=(14, 4))
plt.plot(sample.index, sample["traffic_volume"])
plt.title("traffic_volume, 3 недели в августе 2018")
plt.show()


In [ ]:
tmp = df.copy()
tmp["weekday"] = tmp.index.weekday
tmp["hour"] = tmp.index.hour

pivot = tmp.groupby(["weekday", "hour"])["traffic_volume"].mean().unstack()
plt.figure(figsize=(14, 5))
plt.imshow(pivot, aspect="auto", cmap="viridis")
plt.colorbar(label="mean traffic_volume")
plt.yticks(range(7), ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
plt.xlabel("hour of day")
plt.title("Средняя загрузка по дню недели и часу")
plt.show()


Видно чёткий будний профиль с двумя пиками (утро/вечер) и заметно более низкую, растянутую по
времени загрузку по выходным. Это подтверждает, что стоит использовать недельную сезонность
(24*7 часов) при моделировании.
